In [503]:
# ✅ MAMBA convolucional con definición explícita de kernel 
import torch
import random
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
# import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from torch.nn.utils.rnn import pad_sequence
from collections import defaultdict
from torch.utils.data import TensorDataset, DataLoader

# 📌 1. Seed para reproducibilidad
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# 📌 2. Forzar determinismo
#torch.backends.cudnn.deterministic = True
#torch.backends.cudnn.benchmark = False
#torch.use_deterministic_algorithms(True)

In [504]:
# 1. Carga de datos
def load_and_preprocess(file):
    df = pd.read_csv(file)
    df = df.sort_values(by=["caseid", "end_timestamp"])
    df["end_timestamp"] = pd.to_datetime(df["end_timestamp"])
    df["delta"] = df.groupby("caseid")["end_timestamp"].diff().dt.total_seconds()
    df["delta"] = df["delta"].fillna(0)
    # df["delta"] = np.log1p(df["delta"].fillna(0)) # Reescalado con log1p
    return df

# Función encargada de extraer las trazas que han sido procesadas por load_and_preprocess
def extract_sequences(df):
    traces = defaultdict(list)
    # df = df.sort_values(["caseid", "end_timestamp"]).reset_index(drop=True) => Isto era o que había antes de ordear os eventos en load_and_preprocess
    for _, row in df.iterrows():
        traces[row["caseid"]].append((row["task"], row["user"], row["delta"]))
    return list(traces.values())

# Función encargada de visualizar las n primeras trazas que han sido procesadas por extract_sequences
def print_first_traces(traces, n=2):
    for i, trace in enumerate(traces[:n]):
        print(f"🔵 Traza {i+1} (caseid = {i}):")
        for task, user, delta in trace:
            print(f"   - Tarefa: {task} | Usuario: {user} | Delta (segundos): {delta:.4f}")
        print()

# Seleccionar los datasets con las particiones de entrenamiento, validación y test
dataset = "bpi12a"
if dataset == "env_permit":
    train_df = load_and_preprocess("/Users/manuel.lama.penin/Desktop/MAMBA/Datasets/env_permit/Folds/train_fold0_variation0_env_permit.csv")
    val_df = load_and_preprocess("/Users/manuel.lama.penin/Desktop/MAMBA/Datasets/env_permit/Folds/val_fold0_variation0_env_permit.csv")
    test_df = load_and_preprocess("/Users/manuel.lama.penin/Desktop/MAMBA/Datasets/env_permit/Folds/test_fold0_variation0_env_permit.csv")

if dataset == "bpi12a":
    train_df = load_and_preprocess("/Users/manuel.lama.penin/Desktop/MAMBA/Datasets/bpic12a/Folds/train_fold2_variation0_BPI_Challenge_2012_A.csv")
    val_df = load_and_preprocess("/Users/manuel.lama.penin/Desktop/MAMBA/Datasets/bpic12a/Folds/val_fold2_variation0_BPI_Challenge_2012_A.csv")
    test_df = load_and_preprocess("/Users/manuel.lama.penin/Desktop/MAMBA/Datasets/bpic12a/Folds/test_fold2_variation0_BPI_Challenge_2012_A.csv")

if dataset == "bpi13cp":
    train_df = load_and_preprocess("/Users/manuel.lama.penin/Desktop/MAMBA/Datasets/bpic13cp/Folds/train_fold4_variation0_BPI_Challenge_2013_closed_problems.csv")
    val_df = load_and_preprocess("/Users/manuel.lama.penin/Desktop/MAMBA/Datasets/bpic13cp/Folds/val_fold4_variation0_BPI_Challenge_2013_closed_problems.csv")
    test_df = load_and_preprocess("/Users/manuel.lama.penin/Desktop/MAMBA/Datasets/bpic13cp/Folds/test_fold4_variation0_BPI_Challenge_2013_closed_problems.csv")

if dataset == "helpdesk":
    train_df = load_and_preprocess("/Users/manuel.lama.penin/Desktop/MAMBA/Datasets/helpdesk/Folds/train_fold1_variation0_Helpdesk.csv")
    val_df = load_and_preprocess("/Users/manuel.lama.penin/Desktop/MAMBA/Datasets/helpdesk/Folds/val_fold1_variation0_Helpdesk.csv")
    test_df = load_and_preprocess("/Users/manuel.lama.penin/Desktop/MAMBA/Datasets/helpdesk/Folds/test_fold1_variation0_Helpdesk.csv")

imprimir= False
if imprimir == True:
    # Visualizar las dos primeras trazas
    train_traces = extract_sequences(train_df)
    print_first_traces(train_traces, n=2)

In [505]:
# 2. Métodos de normalización de tiempos
normalizacion = "estandarizacion"

def buckets(df, n_buckets=10):
    cortes = pd.qcut(train_df["delta"], q=n_buckets, retbins=True, duplicates='drop')[1]
    df["delta"] = pd.cut(df["delta"], bins=cortes, labels=False, include_lowest=True) + 1
    return df

def normalize_tiempos(df, global_min, global_max):
    df["delta"] = (df["delta"] - global_min) / (global_max - global_min + 1e-8)
    return df

def standardize_tiempos(df, global_mean, global_std):
    df["delta"] = (df["delta"] - global_mean) / (global_std + 1e-8)
    return df

if normalizacion == "buckets":
    train_df = buckets(train_df)
    val_df = buckets(val_df)
    test_df = buckets(test_df)

if normalizacion == "min-max":
    all_train_val = pd.concat([train_df, val_df])
    global_min = all_train_val["delta"].min()
    global_max = all_train_val["delta"].max()

    train_df = normalize_tiempos(train_df, global_min, global_max)
    val_df = normalize_tiempos(val_df, global_min, global_max)
    test_df = normalize_tiempos(test_df, global_min, global_max)

if normalizacion == "estandarizacion":
    all_train_val = pd.concat([train_df, val_df])
    global_mean = all_train_val["delta"].mean()
    global_std = all_train_val["delta"].std()

    train_df = standardize_tiempos(train_df, global_mean, global_std)
    val_df = standardize_tiempos(val_df, global_mean, global_std)
    test_df = standardize_tiempos(test_df, global_mean, global_std)

if imprimir == True:
    # Visualizar las dos primeras trazas
    train_traces = extract_sequences(train_df)
    print_first_traces(train_traces, n=2)
    
    # Obtiene los buckets
    print(train_df["delta"].value_counts().sort_index())
    
    # Obtiene el valor máximo
    max_tempo = max(
        evento[2]
        for traza in train_traces
        for evento in traza
        if evento[2] is not None
    )
    
    print(f"Máximo de las trazas después de la normalización: {max_tempo:.4f}")

In [506]:
# 3. Procesado de trazas con recursos y tiempos
def generate_io_pairs(traces):
    X, X_user, X_delta, y = [], [], [], []
    for trace in traces:
        for i in range(1, len(trace)):
            acts = [a for a, _, _ in trace[:i]]
            users = [u for _, u, _ in trace[:i]]
            deltas = [d for _, _, d in trace[:i]]
            X.append(acts)
            X_user.append(users)
            X_delta.append(deltas)
            y.append(trace[i][0])
    return X, X_user, X_delta, y

train_traces = extract_sequences(train_df)
val_traces = extract_sequences(val_df)
test_traces = extract_sequences(test_df)

train_x_raw, train_u_raw, train_d_raw, train_y_raw = generate_io_pairs(train_traces)
val_x_raw, val_u_raw, val_d_raw, val_y_raw = generate_io_pairs(val_traces)
test_x_raw, test_u_raw, test_d_raw, test_y_raw = generate_io_pairs(test_traces)

if imprimir == True:
    # Visualizar los datos de entrenamiento con el prefijo y la actividad a predecir
    for i in range(4):
        print(f"🔹 Par {i+1}:")
        print(f"   - Actividades (X): {train_x_raw[i]}")
        print(f"   - Usuarios (U): {train_u_raw[i]}")
        print(f"   - Deltas (D): {train_d_raw[i]}")
        print(f"   - Predición (y): {train_y_raw[i]}")
        print("-" * 50)

In [507]:
# 4. Introducir padding para unificar el tamaño del prefijo
# Clase para codificar las actividades sin reordenalas, evitando que PAD sea la última actividad codificada
class CustomLabelEncoder:
    def __init__(self):
        self.classes_ = []
        self.class_to_idx = {}

    def fit(self, values_list):
        self.classes_ = values_list
        self.class_to_idx = {c: i for i, c in enumerate(values_list)}

    def transform(self, values):
        return [self.class_to_idx[v] for v in values]

    def inverse_transform(self, indices):
        return [self.classes_[i] for i in indices]

# Crear los codificadores para las actividades y recursos
all_activities = sorted(set(train_df["task"].tolist() + val_df["task"].tolist() + test_df["task"].tolist()))
all_users = sorted(set(train_df["user"].tolist() + val_df["user"].tolist() + test_df["user"].tolist()))

activity_encoder = CustomLabelEncoder()
activity_encoder.fit(["PAD"] + all_activities)
user_encoder = CustomLabelEncoder()
user_encoder.fit(["PAD"] + all_users)

# Codifica las actividades y recursos como enteros para poder introducirlos en la red neuronal 
def encode_data(acts, users, deltas, outputs):
    x_enc = [activity_encoder.transform(seq) for seq in acts]
    u_enc = [user_encoder.transform(seq) for seq in users]
    y_enc = activity_encoder.transform(outputs)
    return x_enc, u_enc, deltas, y_enc

train_x_enc, train_u_enc, train_d_enc, train_y_enc = encode_data(train_x_raw, train_u_raw, train_d_raw, train_y_raw)
val_x_enc, val_u_enc, val_d_enc, val_y_enc = encode_data(val_x_raw, val_u_raw, val_d_raw, val_y_raw)
test_x_enc, test_u_enc, test_d_enc, test_y_enc = encode_data(test_x_raw, test_u_raw, test_d_raw, test_y_raw)

# Función encargada de realizar el padding sobre todos los prefijos de los conjuntos de datos
def pad_all(x, u, d, pad=0):
    max_len = max(len(seq) for seq in x)
    x_pad = [torch.tensor([pad]*(max_len - len(s)) + s) for s in x]
    u_pad = [torch.tensor([pad]*(max_len - len(s)) + s) for s in u]
    d_pad = [torch.tensor([0.0]*(max_len - len(s)) + s, dtype=torch.float32) for s in d]
    return torch.stack(x_pad), torch.stack(u_pad), torch.stack(d_pad)

train_x, train_u, train_d = pad_all(train_x_enc, train_u_enc, train_d_enc)
val_x, val_u, val_d = pad_all(val_x_enc, val_u_enc, val_d_enc)
test_x, test_u, test_d = pad_all(test_x_enc, test_u_enc, test_d_enc)

if imprimir == True:
    # Visualizar las actividades y su codificación en enteros
    print("\n👉 Codificación de las actividades (Nombre → Número):")
    for idx, activity in enumerate(activity_encoder.classes_):
        print(f"Actividad '{activity}' → Código {idx}")
    
    # Visualizar los prefijos después de haber realizado padding
    # Visualizar os 10 primeiros prefixos despois do padding
    for i in range(8):
        print(f"🔹 Prefijo {i+1}:")
        
        actividades = train_x[i].tolist()  # Pasámolo de tensor a lista
        usuarios = train_u[i].tolist()
        tempos = train_d[i].tolist()
    
        print(f"  Actividades codificadas: {actividades}")
        print(f"  Recursos codificados:    {usuarios}")
        print(f"  Tiempos:                 {tempos}")
        print()

train_y = torch.tensor(train_y_enc)
val_y = torch.tensor(val_y_enc)
test_y = torch.tensor(test_y_enc)

In [508]:
# 5. Funciones auxiliares
# Función sinusoidal para representation de tiempos
def sinusoidal_encoding(delta, d_model):
    position = delta.unsqueeze(-1)  # [B, L, 1]
    div_term = torch.exp(torch.arange(0, d_model, 2, device=delta.device) * (-np.log(10000.0) / d_model))
    pe = torch.zeros(*position.shape[:-1], d_model, device=delta.device)
    pe[..., 0::2] = torch.sin(position * div_term)
    pe[..., 1::2] = torch.cos(position * div_term)
    return pe  # [B, L, D]

# Clase Time2Vec para representación de tiempos
class Time2Vec(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        
        self.linear = nn.Linear(1, 1)  # La primera componente es lineal (como un "skip connection")
        # Las otras d_model - 1 componentes son sinusoides aprendibles
        self.w = nn.Parameter(torch.randn(1, d_model - 1))  # Frecuencias
        self.b = nn.Parameter(torch.randn(1, d_model - 1))  # Fases

    def forward(self, t):
        """
        t: tensor de forma [B, L] (batch, longitud)
        devuelve: tensor de forma [B, L, out_features]
        """
        lin = self.linear(t)  # Componente lineal => [B, L, 1]
        sin = torch.sin(t * self.w + self.b)  # Componentes senoidales => [B, L, d_model - 1]
        return torch.cat([lin, sin], dim=-1)  # [B, L, d_model]

# Función para representación de la matriz A de la Mamba
def hippo(N):
    A = torch.zeros((N, N))
    for n in range(N):
        for k in range(N):
            if n > k:
                A[n, k] = (2*n + 1)**0.5 * (2*k + 1)**0.5
            elif n == k:
                A[n, k] = n + 1
            else:
                A[n, k] = 0.0
    return A

# Técnica de normalización
class RMSNormGated(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d_model))
        self.gate = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)  # [B, L, 1]
        normed = x / rms  # RMSNorm
        gated = normed * torch.sigmoid(self.gate)  # Aplicamos o gate
        return gated * self.scale  # Reescalamos

# 📌 Umbral con Straight-Through Estimator (STE)
class ThresholdSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input, threshold):
        return (input > threshold).float()

    @staticmethod
    def backward(ctx, grad_output):
        # STE: deixa pasar o gradiente sen cambios
        return grad_output, None

In [509]:
# 🔧 6. Versión convolucional de la Mamba
class MambaConvolucional(nn.Module):
    # Construcción de A como matriz HiPPO.
    def __init__(self, d_model, d_state, kernel_size, vocab_size, user_vocab_size, user_dim=64, p_dropout=0.3, apilada=0):
        super().__init__()
        self.apilada = apilada
        self.d_model = d_model
        self.d_state = d_state
        self.d_delta = d_model

        # Embeddings de actividades y recursos
        self.activity_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.resource_embedding = nn.Embedding(user_vocab_size, user_dim, padding_idx=0)
        self.time2vec = Time2Vec(d_model=self.d_model)
        nn.init.xavier_uniform_(self.activity_embedding.weight)
        nn.init.xavier_uniform_(self.resource_embedding.weight)
        with torch.no_grad():
            self.activity_embedding.weight[0].fill_(0)
            self.resource_embedding.weight[0].fill_(0)

        # Parámetros aprendibles de MAMBA
        self.kernel_size = kernel_size  # Número de térrminoos: CB, CAB, CA²B...
        self.kernel_mask = nn.Parameter(torch.randn(self.kernel_size))  # [K]
        self.delta_lstm = nn.LSTM(input_size=self.d_model, hidden_size=32, batch_first=True)
        self.delta_linear = nn.Sequential(nn.Linear(32, 1), nn.Softplus())
        self.register_buffer("A", hippo(d_state)) # Alternativa: self.A = nn.Parameter(torch.randn(d_model, d_state))
        self.B = nn.Parameter(torch.randn(d_model, d_state))
        self.C = nn.Parameter(torch.randn(d_model, d_state))
        self.D = nn.Parameter(torch.ones(d_model))  # Para conexión residual
        nn.init.uniform_(self.A, -0.5, -0.01)  # Pequenos valores negativos
        nn.init.normal_(self.B, std=0.05)
        nn.init.normal_(self.C, std=0.05)
        nn.init.constant_(self.D, 1.0)

        # Capas de cambio de dimensionalidad
        self.attn_pool = nn.MultiheadAttention(embed_dim=d_model, num_heads=4, batch_first=True)
        self.embeddings_entrada = nn.Linear(d_model, d_model)
        if self.apilada == 0: 
            self.out_mamba = nn.Linear(d_model, vocab_size)
        else: 
            self.out_mamba = nn.Linear(d_model, d_model)            
        self.dim_recurso_a_modelo = nn.Linear(user_dim, d_model)
        
        # Capas de normalización: dropout y RMSNormGated
        self.dropout = nn.Dropout(p_dropout)
        self.norm = RMSNormGated(d_model)
    
    # Discretización de matrices A y B: exp(ΔA), (exp(ΔA)−I)A⁻¹B.
    def discretizar_A_B(self, A, B, delta):
        # mps no tiene sorporte para matrix_exp
        if A.device.type == 'mps':
            A_bar = torch.matrix_exp(delta.cpu() * A.cpu()).to(A.device)
        else:
            A_bar = torch.matrix_exp(delta * A)  # exp(ΔA)
        
        I = torch.eye(A.size(0), device=A.device)
        A_inv = torch.linalg.inv(A)
        B_bar = (A_bar - I) @ A_inv @ B
        return A_bar, B_bar
    
    def build_kernel(self, delta):
        K = []
        A, B = self.discretizar_A_B(self.A, self.B, delta)
        AB = B.clone()  # Comezamos con B
        # Construcción del kernel
        for _ in range(self.kernel_size):
            K_t = torch.sum(self.C * AB, dim=1)  # Paso 1: C * AB => [D]
            K.append(K_t)
            AB = A * AB  # Paso 2: Actualizar AB

        # Resultado: [K, D]
        mask = torch.sigmoid(self.kernel_mask)  # [K]
        for i in range(self.kernel_size):
            K[i] = K[i] * mask[i]

        K = K[::-1]
        kernel = torch.stack(K, dim=0)  # [kernel_size, d_model]
        return kernel

    def forward(self, entrada, recurso=None, tiempo=None):
        if self.apilada != 2:
            B_size, L = entrada.shape
            D = self.d_model
            
            # Embeddings
            emb_entrada = self.activity_embedding(entrada)   # [B, L, D]
            emb_recurso = self.resource_embedding(recurso)   # [B, L, user_dim]
            emb_tiempo = self.time2vec(tiempo.unsqueeze(-1)) # [B, L, D]
            emb_entrada = self.embeddings_entrada(emb_entrada + emb_recurso + emb_tiempo)
            
            # Procesamiento de la entrada
            _, (h_n, _) = self.delta_lstm(emb_entrada)  # h_n: [1, B, 64]
            delta = self.delta_linear(h_n.squeeze(0)).squeeze(-1)  # [B]
            
            # Construir kernels: uno para cada uno de los elementos del batch → [B, K, D]
            kernels = []
            for b in range(B_size):
                delta_b = delta[b]
                kernel_b = self.build_kernel(delta_b)  # [K, D]
                kernels.append(kernel_b)
            kernel = torch.stack(kernels, dim=0)  # [B, K, D]

            # Aplicar el kernel a las entradas de batch
            out = []    
            for t in range(L):
                val = torch.zeros((B_size, D), device=emb_entrada.device)
                for k in range(self.kernel_size):
                    if t - k >= 0:
                        val += emb_entrada[:, t-k, :] * kernel[:, k, :]
                out.append(val)

            # Preparar la salia de la red convolucional
            out = torch.stack(out, dim=1)  # [B, L, D]
            out = self.dropout(out)
            out = self.norm(out)
            out = out * self.D.view(1, 1, -1)
            return self.out_mamba(out[:, -1, :])
        else:
            kernel = self.build_kernel(self.delta)  # [K, D]
            B, D = entrada.shape
            entrada = entrada.unsqueeze(1).repeat(1, self.kernel_size, 1)  # [B, K, D]
            k = kernel.unsqueeze(0).expand(B, -1, -1)  # [B, K, D]
            out = torch.sum(entrada * k, dim=1)  # [B, D]
            out = out * self.D.view(1, -1)
            return self.out_mamba(out)  # [B, D]

In [510]:
# 7. Clase que apila varias MambaConvolucional
class MambaStack(nn.Module):
    def __init__(self, d_model, d_state, kernel_size, vocab_size, user_vocab_size, 
                 num_layers=3, user_dim=64, p_dropout=0.3):
        super().__init__()
        
        self.d_model = d_model
        self.num_layers = num_layers
        
        # Primeira capa: recibe índices (entrada, recurso, tempo)
        self.input_layer = MambaConvolucional(
            d_model=d_model,
            d_state=d_state,
            kernel_size=kernel_size,
            vocab_size=vocab_size,
            user_vocab_size=user_vocab_size,
            user_dim=user_dim,
            p_dropout=p_dropout,
            apilada=1
        )
        
        # Capas intermedias: reciben embeddings [B, D]
        self.blocks = nn.ModuleList([
            MambaConvolucional(
                d_model=d_model,
                d_state=d_state,
                kernel_size=kernel_size,
                vocab_size=vocab_size,
                user_vocab_size=user_vocab_size,
                user_dim=user_dim,
                p_dropout=p_dropout,
                apilada=2
            )
            for _ in range(num_layers - 1)  # resto de capas
        ])
        
        # Normalización despois de cada bloque
        self.norms = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(num_layers)
        ])
        
        # Capa de saída final para predición
        self.out_final = nn.Linear(d_model, vocab_size)

    def forward(self, entrada, recurso, tiempo):
        # Paso 1: pasar pola capa de entrada
        x = self.input_layer(entrada, recurso, tiempo)  # [B, D]
        x = self.norms[0](x)  # Normalizar despois da primeira capa

        # Paso 2: pasar polos bloques apilados
        for i, block in enumerate(self.blocks):
            residual = x
            x = block(x)           # [B, D]
            x = x + residual       # Conexión residual
            x = self.norms[i+1](x) # Normalización

        # Paso 3: predición final
        out = self.out_final(x)  # [B, vocab_size]
        return out

In [511]:
# 8. Versión recurrente de la Mamba
class MambaRecurrente(nn.Module):
    def __init__(self, d_model, d_state, vocab_size, user_vocab_size, user_dim):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_delta = d_model
        print("Dimensión del modelo/entrada: ", d_model)
        print("Dimensión del estado interno: ", d_state)
        print("Dimensión de recursos: ", user_dim)
        print("Dimensión del tiempo: ", d_model)

        # Embeddings de actividades y recursos
        print("Número de actividades: ", vocab_size)
        print("Número de recursos: ", user_vocab_size)
        self.activity_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.resource_embedding = nn.Embedding(user_vocab_size, user_dim, padding_idx=0)
        self.time2vec = Time2Vec(d_model=self.d_model)
        nn.init.xavier_uniform_(self.activity_embedding.weight)
        nn.init.xavier_uniform_(self.resource_embedding.weight)
        with torch.no_grad():
            self.activity_embedding.weight[0].fill_(0)
            self.resource_embedding.weight[0].fill_(0)

        # Parámetros SSM
        self.delta = nn.Parameter(torch.tensor(1.0))
        self.A = nn.Parameter(torch.randn(d_model, d_state))
        self.B = nn.Parameter(torch.randn(d_model, d_state))
        self.C = nn.Parameter(torch.randn(d_model, d_state))
        self.D = nn.Parameter(torch.ones(d_model))  # Para conexión residual
        nn.init.uniform_(self.A, -0.5, -0.01)  # Pequenos valores negativos
        nn.init.normal_(self.B, std=0.05)
        nn.init.normal_(self.C, std=0.05)
        nn.init.constant_(self.D, 1.0)
        
        # Capas de cambio de dimensionalidad
        self.embeddings_entrada = nn.Linear(d_model, d_model)
        self.output_layer = nn.Linear(d_model, vocab_size)
        self.proj_u = nn.Linear(d_model, d_model)
        self.proj_v = nn.Linear(d_model, d_model)

    def discretizar_A_B(self, A, B, delta):
        A_bar = torch.matrix_exp(delta * A)
        I = torch.eye(A.size(0), device=A.device)
        A_inv = torch.linalg.inv(A)
        B_bar = (A_bar - I) @ A_inv @ B
        return A_bar, B_bar
    
    def forward(self, entrada, recurso, tiempo): # x_seq: [B, L, D]
        device = self.A.device
        entrada = entrada.to(device)

        # Embeddings 
        emb_entrada = self.activity_embedding(entrada)  # [B, L, D]
        emb_recurso = self.resource_embedding(recurso)   # [B, L, user_dim]
        emb_tiempo = self.time2vec(tiempo.unsqueeze(-1)) # [B, L, D]
        
        # Procesamiento de la entrada
        emb_entrada = self.embeddings_entrada(emb_entrada + emb_recurso + emb_tiempo)
        
        B, L = entrada.shape
        D = self.d_model
        A_bar, B_bar = self.discretizar_A_B(self.A, self.B, self.delta)  # [D, S]
        ssm_state = torch.zeros(B, self.d_model, A_bar.shape[1], device=device)  # [B, D, S]

        for t in range(L):
            x_t = emb_entrada[:, t, :]     # [B, D]
            # u = F.silu(self.proj_u(x_t))   # [B, D]
            v = F.silu(self.proj_v(x_t))   # [B, D]

            # Estado recurrente
            ssm_state = torch.matmul(ssm_state, A_bar) + v.unsqueeze(-1) * B_bar.unsqueeze(0)  # [B, D, S]
            y = torch.sum(ssm_state * self.C.unsqueeze(0), dim=-1) + self.D * v  # [B, D]
            # z = y * u                                      # Selective multiplicative gate

        logits = self.output_layer(y)  # [B, vocab_size]
        return logits

In [512]:
# ADESTRAMENTO DA REDE
if torch.cuda.is_available(): device = torch.device("cuda")
else: device = torch.device("cpu")
if torch.backends.mps.is_available(): device = torch.device("mps")
print(device)

device = "cpu"

vocab_size = len(activity_encoder.classes_)
user_vocab_size = len(user_encoder.classes_)

def dim_embedding_adaptativa(vocab_size, min_dim=32, max_dim=256):
    dim = int(1.6 * math.sqrt(vocab_size))
    return 32
    # return max(min_dim, min(dim, max_dim))

print("Tamaño do embedding de recurso: ", dim_embedding_adaptativa(user_vocab_size))
print("Tamaño do embedding de actividade: ", dim_embedding_adaptativa(vocab_size))
print(vocab_size)
print(user_vocab_size)

model_size= dim_embedding_adaptativa(vocab_size)
user_size= dim_embedding_adaptativa(user_vocab_size)
model = MambaConvolucional(model_size, model_size, 5, vocab_size, user_vocab_size, user_size, 0.5, 0).to(device)
# model = MambaStack(model_size, model_size, 6, vocab_size, user_vocab_size, 2, user_size).to(device)
# model = MambaRecurrente(model_size, model_size, vocab_size, user_vocab_size, user_size)

loss_fn = nn.CrossEntropyLoss()
#optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
optimizer = torch.optim.AdamW(model.parameters(), lr= 0.001, weight_decay=0.01)

######
val_x, val_u, val_d, val_y = val_x.to(device), val_u.to(device), val_d.to(device), val_y.to(device)
test_x, test_u, test_d, test_y = test_x.to(device), test_u.to(device), test_d.to(device), test_y.to(device)

# 1. Crear DataLoader para mini-batches
def get_dataloader(x, u, d, y, batch_size):
    dataset = TensorDataset(x, u, d, y)
    return DataLoader(dataset, batch_size=batch_size, worker_init_fn=lambda _: np.random.seed(seed), shuffle=True)
train_loader = get_dataloader(train_x, train_u, train_d, train_y, batch_size=32)

# 2. Resto igual
best_moving_avg = float('inf')
best_val_loss = float('inf')
best_val_acc= 0.0;
val_acc= 0.0;
patience = 12
counter = 0
best_model_state = None
early_stopping= "low_val_loss"
window_size = 5
train_losses, val_losses = [], []
val_losses = []
evolucion_mask = []

for epoch in range(100):
    model.train()
    epoch_loss = 0
    for x_batch, u_batch, d_batch, y_batch in train_loader:
        x_batch, u_batch, d_batch, y_batch = x_batch.to(device), u_batch.to(device), d_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(x_batch, u_batch, d_batch)
        loss = loss_fn(pred, y_batch)
        loss.backward()
        #torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Avaliación do modelo durante o adestramento
    model.eval()
    with torch.no_grad():
        val_pred = model(val_x, val_u, val_d)
        val_loss = loss_fn(val_pred, val_y).item()
        val_losses.append(val_loss)

        # Calcular as predicións para partición de adestramento e validación
        val_preds = val_pred.argmax(dim=-1)
        val_acc = (val_preds == val_y).float().mean().item()

        # Mostrar visualizacións dos resultados intermedios
        print(f"Época {epoch+1} | Perda train: {avg_train_loss:.4f} | Perda val: {val_loss:.4f} | Accuracy val: {val_acc:.4f}")
 
    # Aplicar early stopping
    if early_stopping == "low_val_loss":
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict()
            counter = 0
    if early_stopping == "best_val_model":
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict()
            counter = 0
    if early_stopping == "avg_val_loss":
        if len(val_losses) >= window_size: moving_avg = sum(val_losses[-window_size:]) / window_size
        else: moving_avg = sum(val_losses) / len(val_losses)
        print(f"Época {epoch+1} | Val_loss: {val_loss:.4f} | Media móbil: {moving_avg:.4f}")
        if moving_avg < best_moving_avg:
            print(f"Media de window: {moving_avg}")
            best_moving_avg = moving_avg
            best_model_state = model.state_dict()
            counter = 0
    counter += 1
    if counter >= patience:
        print(f"⏹️ Early stopping na época {epoch+1} | Mellor val_loss: {best_val_loss:.4f}")
        break

if best_model_state:
    model.load_state_dict(best_model_state)

mps
Tamaño do embedding de recurso:  32
Tamaño do embedding de actividade:  32
11
62
Época 1 | Perda train: 0.8266 | Perda val: 0.5158 | Accuracy val: 0.7526
Época 2 | Perda train: 0.5657 | Perda val: 0.5033 | Accuracy val: 0.7219
Época 3 | Perda train: 0.5436 | Perda val: 0.5101 | Accuracy val: 0.7510
Época 4 | Perda train: 0.5376 | Perda val: 0.4995 | Accuracy val: 0.7573
Época 5 | Perda train: 0.5320 | Perda val: 0.5046 | Accuracy val: 0.7534
Época 6 | Perda train: 0.5299 | Perda val: 0.4938 | Accuracy val: 0.7528
Época 7 | Perda train: 0.5298 | Perda val: 0.4920 | Accuracy val: 0.7234
Época 8 | Perda train: 0.5263 | Perda val: 0.4875 | Accuracy val: 0.7567
Época 9 | Perda train: 0.5242 | Perda val: 0.5007 | Accuracy val: 0.7578
Época 10 | Perda train: 0.5228 | Perda val: 0.4962 | Accuracy val: 0.7574
Época 11 | Perda train: 0.5233 | Perda val: 0.4891 | Accuracy val: 0.7550
Época 12 | Perda train: 0.5221 | Perda val: 0.4978 | Accuracy val: 0.7568
Época 13 | Perda train: 0.5210 | Per

In [513]:
# AVALIACIÓN FINAL
model.eval()
with torch.no_grad():
    probs = model(test_x, test_u, test_d)
    preds = probs.argmax(dim=-1)
    acc = (preds == test_y).float().mean().item()
    print(f"✅ Precisión no test: {acc:.4f}")

# 7. Visualización de perdas
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Validación")
plt.title("Evolución da perda")
plt.xlabel("Épocas")
plt.ylabel("Perda")
plt.legend()
plt.grid(True)
plt.show()

✅ Precisión no test: 0.7544


NameError: name 'plt' is not defined